# Semana 3 — Mecánica de clasificación (logística, vinos y árboles)

Este cuaderno es el satélite de *mecánica* de clasificación de la semana. Recorre tres familias
de modelos y les aplica una **medición honesta**:

1. **Regresión logística** (`datasets/ejemplos.csv`) — primero programada desde cero con descenso
   de gradiente, para ver qué hay dentro, y después con `LogisticRegression` de `sklearn`, que es
   la que usaremos en adelante. Ahora con partición y métricas en vez de mirar la frontera *a ojo*.
2. **El marco de evaluación** se **define aquí**, justo cuando ya hay predicciones que
   medir (matriz de confusión, precisión/recall/F1, umbral, ROC/AUC).
3. **Vinos** (`datasets/winequality-white.csv`) — logística de `sklearn` sobre datos reales. Termina con
   un **ejercicio de interpretación** de las métricas, incluidas la curva ROC y el AUC. Es donde la evaluación **muerde**: la exactitud engaña.
4. **Árboles** (`datasets/simpleclass_dataset.txt`) — árbol con `sklearn`, partición y **matriz de
   confusión multiclase**.

> Primero se definen las métricas sobre un caso sintético y fácil (la logística sobre
> `ejemplos.csv`). Después se aplica esa misma medición a datos reales con clases desbalanceadas
> (vinos) y a un problema de cuatro clases (árboles).

## 0. Preparación

In [ ]:
%matplotlib inline
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, precision_recall_curve)


def dibuja_regiones(clasificador, puntos, clases, ax, titulo):
    """Colorea la región que un clasificador 2D ya entrenado asigna a cada clase y dibuja los puntos."""
    x1_min = puntos[:, 0].min() - 0.5
    x1_max = puntos[:, 0].max() + 0.5
    x2_min = puntos[:, 1].min() - 0.5
    x2_max = puntos[:, 1].max() + 0.5
    valores_x1 = np.linspace(x1_min, x1_max, 300)
    valores_x2 = np.linspace(x2_min, x2_max, 300)
    malla_x1, malla_x2 = np.meshgrid(valores_x1, valores_x2)
    puntos_malla = np.column_stack([malla_x1.ravel(), malla_x2.ravel()])
    clase_malla = clasificador.predict(puntos_malla)
    clase_malla = clase_malla.reshape(malla_x1.shape)
    ax.contourf(malla_x1, malla_x2, clase_malla, alpha=0.2, cmap="tab10")
    ax.scatter(puntos[:, 0], puntos[:, 1], c=clases, s=6, cmap="tab10")
    ax.set_title(titulo)
    ax.set_xticks([])
    ax.set_yticks([])

## 1. Regresión logística: desde cero y con `sklearn`

Empezamos con `datasets/ejemplos.csv`: dos coordenadas y una clase (0/1), con los puntos de una
clase formando un círculo dentro de los de la otra, así que una recta no los separa. Lo usamos
para ver cómo funciona la regresión logística y para definir las métricas de evaluación.
Fíjate en el balance de clases: **no está al 50 %**.

In [ ]:
tabla_ejemplos = pd.read_csv('../datasets/ejemplos.csv')   # columnas: x1, x2, clase (0/1)
tabla_ejemplos.head()

In [ ]:
puntos = tabla_ejemplos[['x1', 'x2']].to_numpy()
clases = tabla_ejemplos['clase'].to_numpy()
clases = clases.astype(int)

n_puntos = len(clases)
fraccion_clase_1 = clases.mean()
print(f"n = {n_puntos} | clase 1 = {fraccion_clase_1:.1%}  (desbalanceado)")

es_clase_0 = clases == 0
es_clase_1 = clases == 1
fig, ax = plt.subplots(figsize=(4.4, 4.0))
ax.scatter(puntos[es_clase_0, 0], puntos[es_clase_0, 1], s=6, c="#C0504D", marker="x", label="clase 0")
ax.scatter(puntos[es_clase_1, 0], puntos[es_clase_1, 1], s=6, c="#1F497D", label="clase 1")
ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.legend(fontsize=8)
ax.axis("equal")
ax.set_title("ejemplos.csv")
plt.tight_layout()
plt.show()

Las piezas **desde cero**: la sigmoide y el **descenso de gradiente de la Semana 2**, con el
gradiente que corresponde a la regresión logística. Solo cambian dos cosas:

- la predicción pasa por la sigmoide: h = σ(Xβ), una probabilidad entre 0 y 1;
- el coste que se registra en el historial es la **entropía cruzada**, no el MSE.

El gradiente tiene la misma forma que en la Semana 2, Xᵀ(h − y)/n, con h en lugar de Xβ.
Te las damos programadas: léelas para entender qué hace la regresión logística por dentro; no
tienes que programarlas.

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def descenso_gradiente(X, y, alfa, n_iter):
    """Descenso de gradiente vectorizado para regresión logística (entropía cruzada).
    Devuelve los coeficientes finales y el historial de coste por iteración."""
    n_datos = X.shape[0]  # número de datos (filas)
    n_coef = X.shape[1]   # número de coeficientes (columnas)
    beta = np.zeros(n_coef)
    historial = np.zeros(n_iter)
    eps = np.finfo(float).eps                     # evita log(0)
    for iteracion in range(n_iter):
        h = sigmoid(X @ beta)                     # probabilidad estimada de clase 1
        error = h - y
        log_verosimilitud = y * np.log(h + eps) + (1 - y) * np.log(1 - h + eps)
        historial[iteracion] = -np.mean(log_verosimilitud)   # entropía cruzada actual
        gradiente = (X.T @ error) / n_datos       # gradiente de la entropía cruzada
        beta = beta - alfa * gradiente            # paso de descenso
    return beta, historial

**Matriz de diseño en espacio extendido** (frontera curva): `[x1, x2, x1·x2, x1², x2²]`.
Y ahora lo importante: **partición honesta**. Estos datos son i.i.d. (cada punto es independiente
de los demás), así que un reparto al azar 70/30 vale.

Repartimos con `train_test_split` de `sklearn`: `test_size=0.3` deja el 30 % para el test y
`random_state=0` hace que el reparto al azar sea siempre el mismo.

Para que el descenso de gradiente converja, **estandarizamos** las columnas con `StandardScaler`,
ajustado **solo con el train**, como en la Semana 2. Después añadimos delante una columna de unos
para el término independiente.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `puntos`, `clases` | coordenadas (x1, x2) y clase (0/1) de cada punto de `ejemplos.csv` |
| `puntos_train`, `puntos_test` | puntos de cada parte (reparto al azar 70/30) |
| `clases_train`, `clases_test` | clase real de los puntos de cada parte |
| `entradas_train_escaladas`, `entradas_test_escaladas` | matriz de diseño estandarizada con el train, sin la columna de unos |
| `X_train` | `entradas_train_escaladas` con la columna de unos delante |
| `beta_logistica`, `historial_logistica` | coeficientes del descenso de gradiente y entropía cruzada en cada iteración |
| `logistica` | la `LogisticRegression` de `sklearn` entrenada |
| `probabilidad_test`, `prediccion_test` | probabilidad de clase 1 y clase predicha (umbral 0,5) en el test |

In [ ]:
def matriz_diseno(puntos):
    x1 = puntos[:, 0]
    x2 = puntos[:, 1]
    return np.column_stack([x1, x2, x1 * x2, x1**2, x2**2])   # sin columna de unos


puntos_train, puntos_test, clases_train, clases_test = train_test_split(
    puntos, clases, test_size=0.3, random_state=0)
n_train = len(clases_train)
n_test = len(clases_test)

diseno_train = matriz_diseno(puntos_train)
diseno_test = matriz_diseno(puntos_test)
escalador = StandardScaler()
escalador.fit(diseno_train)                        # media y desviación, solo con el train
entradas_train_escaladas = escalador.transform(diseno_train)
entradas_test_escaladas = escalador.transform(diseno_test)

unos_train = np.ones((n_train, 1))                 # columna para el término independiente
X_train = np.hstack([unos_train, entradas_train_escaladas])

beta_logistica, historial_logistica = descenso_gradiente(X_train, clases_train, alfa=0.1, n_iter=20000)
print(f"Entropía cruzada en el train: iteración 0 = {historial_logistica[0]:.3f} | última = {historial_logistica[-1]:.3f}")

fraccion_clase_1_test = clases_test.mean()
print(f"Test: {n_test} puntos | clase 1 = {fraccion_clase_1_test:.1%}")

**Ahora, con `sklearn`.** Programarla desde cero sirve para entender qué hay dentro (sigmoide,
entropía cruzada y descenso de gradiente). En la práctica se usa `LogisticRegression` de
`sklearn`, y es la que usamos **a partir de aquí**.

Dos detalles:

- `sklearn` añade el término independiente por su cuenta: le pasamos las mismas entradas
  estandarizadas, sin la columna de unos.
- Por defecto aplica una **regularización** suave, como el ridge de la Semana 2: el parámetro `C`
  hace el papel de 1/λ (cuanto menor es `C`, más regulariza). Sin ella, en estos datos casi
  separables los coeficientes crecerían sin límite.


In [ ]:
logistica = LogisticRegression(max_iter=5000)
logistica.fit(entradas_train_escaladas, clases_train)   # sklearn añade el término independiente

probabilidades = logistica.predict_proba(entradas_test_escaladas)
probabilidad_test = probabilidades[:, 1]                # columna 1 = probabilidad de clase 1
prediccion_test = (probabilidad_test >= 0.5).astype(int)   # umbral 0,5

## 2. Del ajuste a la evaluación

Ya tenemos `probabilidad_test` (probabilidad de clase 1 en el *test*) y `prediccion_test`
(clase predicha con umbral 0,5).
**Ahora** toca medir. Empezamos por la **matriz de confusión** (clase 1 = "positivo"):

|            | pred = 0 | pred = 1 |
|------------|:--------:|:--------:|
| **real 0** | TN       | FP       |
| **real 1** | FN       | TP       |

Cada casilla cuenta puntos del test según su clase real, $y_i$, y la predicha, $\hat{y}_i$. Con
$\#\{\dots\}$ = «número de puntos que cumplen…»:

- **TP** (*true positives*, verdaderos positivos): son de clase 1 y el modelo dice 1. Aciertos.
  $$\mathrm{TP} = \#\{\,i : y_i = 1,\ \hat{y}_i = 1\,\}$$
- **TN** (*true negatives*, verdaderos negativos): son de clase 0 y el modelo dice 0. Aciertos.
  $$\mathrm{TN} = \#\{\,i : y_i = 0,\ \hat{y}_i = 0\,\}$$
- **FP** (*false positives*, falsos positivos): son de clase 0 y el modelo dice 1. Falsas alarmas.
  $$\mathrm{FP} = \#\{\,i : y_i = 0,\ \hat{y}_i = 1\,\}$$
- **FN** (*false negatives*, falsos negativos): son de clase 1 y el modelo dice 0. Se le escapan.
  $$\mathrm{FN} = \#\{\,i : y_i = 1,\ \hat{y}_i = 0\,\}$$

Entre las cuatro suman todos los puntos del test: TP + TN + FP + FN = n.

«Verdadero» o «falso» dice si el modelo acierta; «positivo» o «negativo», lo que predice. Los aciertos
están en la diagonal. La calculamos y la dibujamos con `sklearn`.

In [ ]:
matriz_confusion = confusion_matrix(clases_test, prediccion_test)     # [[TN, FP], [FN, TP]]
grafico_matriz = ConfusionMatrixDisplay(matriz_confusion, display_labels=["c0", "c1"])
grafico_matriz.plot(cmap="Blues", colorbar=False)
plt.title("Matriz de confusión — logística (sklearn)")
plt.tight_layout()
plt.show()

De la matriz salen las métricas:

- **Exactitud** (*accuracy*) = (TP + TN) / (TP + TN + FP + FN): fracción de **todos** los puntos que
  el modelo clasifica bien, es decir, la diagonal de la matriz entre el total. No distingue clases.
- **Recall** de la clase positiva = TP / (TP + FN): de los que son 1, fracción que el modelo marca
  como 1. También se llama **tasa de verdaderos positivos** (TPR).
- **Tasa de falsos positivos** (FPR) = FP / (FP + TN): de los que son 0, fracción que el modelo marca
  como 1 por error.
- **Precisión** de la clase positiva = TP / (TP + FP): de los que el modelo marca como 1, fracción que
  lo son de verdad.
- **F1** = 2 · precisión · recall / (precisión + recall): la media armónica de las dos. Solo es alta
  si precisión y recall lo son a la vez.

`sklearn` las calcula con `accuracy_score`, `recall_score`, `precision_score` y `f1_score`.

In [ ]:
# === TODO ===  (clase 1 = positiva)
# pista: accuracy_score(clases_test, prediccion_test)
exactitud = ...   # <-- TODO
precision = ...   # <-- TODO
recall = ...      # <-- TODO
f1 = ...          # <-- TODO
print(f"exactitud={exactitud:.3f}  precisión={precision:.3f}  recall={recall:.3f}  F1={f1:.3f}")

In [ ]:
informe = classification_report(clases_test, prediccion_test, target_names=["c0", "c1"], digits=3)
print(informe)
auc = roc_auc_score(clases_test, probabilidad_test)
print("AUC =", round(auc, 3))

La **curva ROC** de este modelo. La interpretarás en el ejercicio del final de la sección 3.

In [ ]:
tasa_fp, tasa_vp, umbrales = roc_curve(clases_test, probabilidad_test)

fig, ax = plt.subplots(figsize=(5, 4.6))
ax.plot(tasa_fp, tasa_vp, label=f"logística (AUC = {auc:.3f})")
ax.plot([0, 1], [0, 1], "k:", lw=1, label="azar (AUC = 0.5)")
ax.set_xlabel("FPR (tasa de falsos positivos)")
ax.set_ylabel("TPR (recall)")
ax.set_title("Curva ROC en el test")
ax.legend(fontsize=8, loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Aquí **casi todo sale perfecto** (matriz casi diagonal, F1 y AUC ≈ 1): `datasets/ejemplos.csv` es
separable con la frontera curva. Es el sitio ideal para *definir* las métricas sin ruido —
**no** para creer que la clasificación siempre es así de fácil. En cuanto pasemos a datos
reales, la exactitud dejará de ser de fiar.

## 3. Vinos: cuando la evaluación muerde

**El dataset.** *Wine Quality* (repositorio UCI; Cortez et al., 2009) recoge **4898 vinos blancos**
de la denominación portuguesa *vinho verde*. De cada vino hay **11 medidas físico-químicas** de
laboratorio (acidez fija y volátil, ácido cítrico, azúcar residual, cloruros, dióxido de azufre
libre y total, densidad, pH, sulfatos y alcohol) y una nota de calidad, `quality`, de 0 a 10, que
es la mediana de al menos tres catadores; en estos datos va de 3 a 9. Las medidas son las
entradas; la nota, la salida. A diferencia de `ejemplos.csv`, son datos reales, y los vinos buenos y
malos se solapan mucho, como verás en los resultados.

Clasificamos vinos blancos como **buenos** (`quality > 5`) o no, con la logística de
`sklearn`. Lo importante: medir sobre los **mismos datos de entrenamiento** y quedarse con **un único número**
engaña. Lo hacemos con partición honesta y miramos lo que ese número esconde.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `caracteristicas_vinos`, `vino_bueno` | las 11 medidas químicas de cada vino y su clase (1 = bueno, 0 = malo) |
| `entradas_train_vinos`, `entradas_test_vinos` | medidas estandarizadas con el train (reparto al azar 70/30) |
| `bueno_train`, `bueno_test` | clase real de los vinos de cada parte |
| `prediccion_base` | la línea base: predecir siempre «bueno» |
| `probabilidad_bueno_test`, `prediccion_test_logistica` | probabilidad de «bueno» y clase predicha por la logística |

In [ ]:
ruta_vinos = "../datasets/winequality-white.csv"          # UCI Wine Quality (blancos), separador ';'
if not os.path.exists(ruta_vinos):
    raise FileNotFoundError(
        "Falta '../datasets/winequality-white.csv' (vinos blancos, sep=';'). Descárgalo en la carpeta datasets.")
vinos = pd.read_csv(ruta_vinos, sep=";")
vinos.head()

In [ ]:
caracteristicas_vinos = vinos.drop(columns="quality")
caracteristicas_vinos = caracteristicas_vinos.to_numpy()
calidad = vinos["quality"].to_numpy()
vino_bueno = (calidad > 5).astype(int)                 # 1 = bueno, 0 = malo

n_vinos = len(vino_bueno)
fraccion_buenos = vino_bueno.mean()
fraccion_malos = 1 - fraccion_buenos
print(f"n = {n_vinos} | buenos = {fraccion_buenos:.1%} | malos = {fraccion_malos:.1%}")

caracteristicas_train_vinos, caracteristicas_test_vinos, bueno_train, bueno_test = train_test_split(
    caracteristicas_vinos, vino_bueno, test_size=0.3, random_state=0)

escalador_vinos = StandardScaler()
escalador_vinos.fit(caracteristicas_train_vinos)     # solo con el train
entradas_train_vinos = escalador_vinos.transform(caracteristicas_train_vinos)
entradas_test_vinos = escalador_vinos.transform(caracteristicas_test_vinos)

prediccion_base = np.ones_like(bueno_test)      # línea base: predecir siempre "bueno" (mayoritaria)
exactitud_base = accuracy_score(bueno_test, prediccion_base)
print(f"Exactitud de la línea base (mayoritaria) = {exactitud_base:.3f}")

**Logística de `sklearn`** (`predict_proba`, `predict`, `score`).

In [ ]:
logistica_vinos = LogisticRegression(max_iter=2000)
logistica_vinos.fit(entradas_train_vinos, bueno_train)
probabilidades_vinos = logistica_vinos.predict_proba(entradas_test_vinos)
probabilidad_bueno_test = probabilidades_vinos[:, 1]     # P(bueno)
prediccion_test_logistica = logistica_vinos.predict(entradas_test_vinos)

informe_vinos = classification_report(bueno_test, prediccion_test_logistica, target_names=["malo", "bueno"], digits=3)
print("classification_report (0 = malo, 1 = bueno):")
print(informe_vinos)

La **matriz de confusión** de la logística en el test. Como en la sección 2, las filas son la clase
real y las columnas, la predicha; ahora la clase 0 es «malo» y la 1, «bueno».

In [ ]:
matriz_confusion_vinos = confusion_matrix(bueno_test, prediccion_test_logistica)   # [[malo→malo, malo→bueno], [bueno→malo, bueno→bueno]]
print(matriz_confusion_vinos)

grafico_matriz_vinos = ConfusionMatrixDisplay(matriz_confusion_vinos, display_labels=["malo", "bueno"])
grafico_matriz_vinos.plot(cmap="Blues", colorbar=False)
plt.title("Matriz de confusión — vinos (logística)")
plt.xlabel("clase predicha")
plt.ylabel("clase real")
plt.tight_layout()
plt.show()

Rellena la tabla comparativa. Mira **el recall de la clase 'malo'** (la minoritaria):
de todos los vinos malos reales, ¿qué fracción caza cada modelo?

In [ ]:
puntuacion_base = np.zeros(len(bueno_test))      # la línea base no da una puntuación que ordene
modelos = {
    "Base (mayor.)": [prediccion_base, puntuacion_base],
    "Logística": [prediccion_test_logistica, probabilidad_bueno_test],
}
print(f"{'modelo':16s}{'exactitud':>10s}{'rec_malo':>10s}{'f1_malo':>9s}{'AUC':>8s}")
for nombre_modelo in modelos:
    prediccion = modelos[nombre_modelo][0]
    puntuacion = modelos[nombre_modelo][1]
    # === TODO ===  (clase 'malo' = 0; AUC con la puntuación continua)
    exactitud = ...     # <-- TODO
    recall_malo = ...   # <-- TODO: recall de la clase 0
    f1_malo = ...       # <-- TODO: F1 de la clase 0
    if puntuacion.std() > 0:
        auc = roc_auc_score(bueno_test, puntuacion)
    else:
        auc = 0.5                                  # una puntuación constante no ordena: AUC = 0,5
    print(f"{nombre_modelo:16s}{exactitud:10.3f}{recall_malo:10.3f}{f1_malo:9.3f}{auc:8.3f}")

In [ ]:
tasa_falsos_positivos, tasa_verdaderos_positivos, umbrales_roc = roc_curve(bueno_test, probabilidad_bueno_test)
precision_curva, recall_curva, umbrales_pr = precision_recall_curve(bueno_test, probabilidad_bueno_test)
auc_logistica = roc_auc_score(bueno_test, probabilidad_bueno_test)

fig, ejes = plt.subplots(1, 2, figsize=(8.4, 3.3))
ejes[0].plot(tasa_falsos_positivos, tasa_verdaderos_positivos)
ejes[0].plot([0, 1], [0, 1], "k:", lw=1)
ejes[0].set_title(f"ROC — AUC = {auc_logistica:.3f}")
ejes[0].set_xlabel("FPR")
ejes[0].set_ylabel("TPR")
ejes[0].grid(alpha=0.3)
ejes[1].plot(recall_curva, precision_curva)
ejes[1].set_title("Precisión–recall (clase bueno)")
ejes[1].set_xlabel("recall")
ejes[1].set_ylabel("precisión")
ejes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Ejercicio: interpretar las métricas

Responde con tus palabras, apoyándote en las salidas de esta sección: el informe, la matriz de
confusión, la tabla frente a la línea base y las curvas. Solo la pregunta 3.5 necesita código.

**Pregunta 3.1 · El informe.** En el `classification_report` de la logística, la clase «malo» tiene
precisión 0,708 y recall 0,469. Explica con palabras, en términos de vinos, qué significa cada
número. ¿Qué indica el soporte (527 y 943)? ¿Por qué el recall medio *macro* (0,680) sale menor que
el ponderado (0,740)? Si lo que te preocupa son los vinos malos, ¿cuál de las dos medias describe
mejor al modelo?

**Tu respuesta:** *escribe aquí*

**Pregunta 3.2 · La matriz de confusión y el coste de cada error.** Para una bodega, ¿qué error es
más caro: colar un vino malo como bueno o rechazar uno bueno? ¿En qué casilla de la matriz está ese
error y qué métrica del informe lo mide? Con el umbral 0,5, de cada 100 vinos que el modelo marca
como buenos, ¿cuántos son en realidad malos?

**Tu respuesta:** *escribe aquí*

**Pregunta 3.3 · Exactitud frente a línea base.** La línea base (predecir siempre «bueno») tiene una
exactitud de 0,641 sin detectar ni un solo vino malo; la logística, de 0,740. ¿Cómo puede la línea
base acertar tanto? ¿Qué métricas de la tabla comparativa la delatan?

**Tu respuesta:** *escribe aquí*

**Pregunta 3.4 · Curva ROC y AUC.** En la curva ROC de esta sección, la clase positiva es «bueno».

- (a) ¿Qué representa cada punto de la curva?
- (b) Lee en la curva qué fracción de vinos buenos detectas si solo aceptas colar el 20 % de los malos.
- (c) El AUC es 0,80. Explícalo como una probabilidad sobre parejas formadas por un vino bueno y uno malo.
- (d) ¿Por qué la línea base tiene AUC 0,5?
- (e) ¿El AUC te dice qué umbral usar?

**Tu respuesta:** *escribe aquí*

**Pregunta 3.5 · Mover el umbral.** `predict` clasifica como «bueno» si la probabilidad de «bueno»
llega a 0,5. Si colar un vino malo es el error caro, ¿subirías o bajarías ese umbral? Compruébalo en
la celda siguiente con el umbral 0,7: ¿qué pasa con el recall de «malo», el de «bueno» y la
exactitud? ¿Qué umbral elegirías tú y por qué?

In [ ]:
umbral = 0.7
# === TODO ===
# Clasifica como «bueno» (1) los vinos cuya probabilidad de «bueno» llega al umbral, y como «malo» (0) el resto.
# pista: (probabilidad_bueno_test >= umbral).astype(int)
prediccion_umbral = ...   # <-- TODO

matriz_umbral = confusion_matrix(bueno_test, prediccion_umbral)
recall_malo_umbral = recall_score(bueno_test, prediccion_umbral, pos_label=0)
recall_bueno_umbral = recall_score(bueno_test, prediccion_umbral, pos_label=1)
exactitud_umbral = accuracy_score(bueno_test, prediccion_umbral)
print(f"Umbral {umbral}")
print(matriz_umbral)
print(f"recall «malo» = {recall_malo_umbral:.3f} | recall «bueno» = {recall_bueno_umbral:.3f} | exactitud = {exactitud_umbral:.3f}")

**Tu respuesta:** *escribe aquí*

## 4. Árboles con evaluación

Cerramos con árboles, sobre `SimpleClass`: **4 clases** en 2D. La
evaluación se generaliza sola: la matriz de confusión pasa de 2×2 a **K×K**. Partición
honesta también aquí, con `train_test_split`.

Variables de esta sección:

| Variable | Qué contiene |
|---|---|
| `puntos_simpleclass`, `clases_simpleclass` | coordenadas (x1, x2) y clase (1 a 4) de cada punto |
| `puntos_train_simpleclass`, `puntos_test_simpleclass` | puntos de cada parte (reparto al azar 70/30) |
| `clases_train_simpleclass`, `clases_test_simpleclass` | clase real de los puntos de cada parte |
| `arbol` | el árbol de decisión entrenado |
| `clases_predichas_simpleclass` | clase que predice el árbol para los puntos del test |

In [ ]:
tabla_simpleclass = pd.read_csv('../datasets/simpleclass_dataset.txt')   # ya es CSV (cabecera x1,x2,clase)
puntos_simpleclass = tabla_simpleclass[['x1', 'x2']].to_numpy()
clases_simpleclass = tabla_simpleclass['clase'].to_numpy()
clases_simpleclass = clases_simpleclass.astype(int)
recuento_clases = np.bincount(clases_simpleclass)
print("clases (conteo):", recuento_clases[1:])

puntos_train_simpleclass, puntos_test_simpleclass, clases_train_simpleclass, clases_test_simpleclass = train_test_split(
    puntos_simpleclass, clases_simpleclass, test_size=0.3, random_state=0)

arbol = DecisionTreeClassifier(max_depth=3, random_state=0)
arbol.fit(puntos_train_simpleclass, clases_train_simpleclass)

fig, ax = plt.subplots(figsize=(4.4, 4.0))
dibuja_regiones(arbol, puntos_test_simpleclass, clases_test_simpleclass, ax, "Árbol (prof=3) — fronteras a trozos")
plt.tight_layout()
plt.show()

La **matriz de confusión K×K** del árbol en el test, con `sklearn`: filas = clase real, columnas =
clase predicha.

In [ ]:
clases_predichas_simpleclass = arbol.predict(puntos_test_simpleclass)
matriz_confusion_k = confusion_matrix(clases_test_simpleclass, clases_predichas_simpleclass)
print("matriz de confusión (filas = clase real 1 a 4, columnas = clase predicha 1 a 4):")
print(matriz_confusion_k)

In [ ]:
prediccion_train_arbol = arbol.predict(puntos_train_simpleclass)
exactitud_train = accuracy_score(clases_train_simpleclass, prediccion_train_arbol)
exactitud_test = accuracy_score(clases_test_simpleclass, clases_predichas_simpleclass)
print(f"exactitud TRAIN = {exactitud_train:.3f} | exactitud TEST = {exactitud_test:.3f}")

El árbol acierta ~100 % **en train y en test**. Un acierto perfecto en train suele ser
señal de alarma (sobreentrenamiento), pero aquí el *held-out* lo **confirma**: el problema es
genuinamente separable. Esa es justo la razón de tener partición — sin ella no podrías
distinguir "de verdad es fácil" de "me lo he aprendido de memoria".

**Atributos categóricos (Play Tennis).** Un ejemplo clásico de árbol con variables
categóricas, vía `get_dummies`, como el de las diapositivas de teoría de árboles. Con 14 filas no hay held-out
razonable; queda como ilustración de la mecánica, no de la evaluación.

In [ ]:
tabla_tenis = pd.DataFrame({
    "Outlook": ["Sunny","Sunny","Overcast","Rain","Rain","Rain","Overcast","Sunny",
                "Sunny","Rain","Sunny","Overcast","Overcast","Rain"],
    "Temp": ["Hot","Hot","Hot","Mild","Cool","Cool","Cool","Mild","Cool","Mild",
             "Mild","Mild","Hot","Mild"],
    "Humidity": ["High","High","High","High","Normal","Normal","Normal","High",
                 "Normal","Normal","Normal","High","Normal","High"],
    "Wind": ["Weak","Strong","Weak","Weak","Weak","Strong","Weak","Weak","Weak",
             "Strong","Strong","Strong","Weak","Strong"],
    "Play": ["No","No","Yes","Yes","Yes","No","Yes","No","Yes","Yes","Yes","Yes","Yes","No"]})
atributos_tenis = tabla_tenis.drop(columns="Play")
entradas_tenis = pd.get_dummies(atributos_tenis)
juega = tabla_tenis["Play"]
arbol_tenis = DecisionTreeClassifier(criterion="entropy")
arbol_tenis.fit(entradas_tenis, juega)
prediccion_tenis = arbol_tenis.predict(entradas_tenis)
exactitud_tenis = accuracy_score(juega, prediccion_tenis)
print("acierto en los 14 casos:", exactitud_tenis)

## 5. Cierre

Tres familias de modelos, un mismo protocolo de medición:

- **Logística desde cero** (`datasets/ejemplos.csv`): definimos las métricas sobre un caso limpio.
- **Vinos**: la exactitud engañaba; F1/recall de la minoría y ROC/AUC lo destaparon.
- **Árboles**: la matriz de confusión se generalizó a K×K y el *held-out* confirmó la
  separabilidad.